<a href="https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/20260824/notebooks/protein_representation_practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 단백질 표현법 실습 — 수작업 표현 vs 학습된 표현

**AI 신약개발 · 단백질 표현법(2026-08-24)**

같은 단백질을 **어떤 벡터로 바꾸느냐**(표현, representation)가 downstream 성능을 좌우합니다.
이 실습은 7/24 강의 흐름 그대로, **실제 단백질 20개(4개 family)** 를 대상으로 세 가지 표현을 만들고 비교합니다.

1. **수작업 표현(hand-crafted)** — one-hot, 아미노산 조성(AAC), 물성(ProtParam)
2. **학습된 표현(learned)** — 단백질 언어모델 **ESM-2** 임베딩
3. **비교** — 같은 과제(family 분류/군집)에서 어느 표현이 더 잘 구분하나 (실측)

> ⚠️ **무-날조 원칙**
> - 서열은 **UniProt에서 실제로 내려받고**, family 라벨은 실제 분류입니다.
> - 정확도·실루엣 등 모든 수치는 이 노트북이 **실제로 계산**한 값입니다(임의 수치 없음).
> - 20개 소규모 데모입니다 — 정식 벤치마크가 아니라 **표현법 감각을 익히는 용도**입니다.

> 🖥️ CPU로도 동작(ESM-2 8M 모델). GPU면 더 빠릅니다.

## 0. 설치 & 환경
Colab에는 torch/transformers/scikit-learn이 대개 사전설치. biopython만 추가로 설치하고, 그래프 한글이 깨지지 않게 한글 폰트를 설정합니다.

In [ ]:
# biopython 설치 (Colab에 없음). torch/transformers/sklearn은 Colab 사전설치.
!pip install -q biopython
# (Colab) matplotlib 한글 폰트 — 없으면 그래프 한글이 □□로 깨짐
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true

import sys, platform
import matplotlib as mpl
import matplotlib.font_manager as fm
_kfonts = [f for f in fm.findSystemFonts() if "Nanum" in f]
for _f in _kfonts:
    fm.fontManager.addfont(_f)
if _kfonts:
    mpl.rcParams["font.family"] = "NanumGothic"
mpl.rcParams["axes.unicode_minus"] = False   # 마이너스 기호 깨짐 방지
print("python", platform.python_version(), "| 한글폰트:", "NanumGothic" if _kfonts else "기본(영문 라벨로 표시)")


## 1. 데이터 — 실제 단백질 20개 (UniProt)

4개 family × 5개. 서열이 명확히 다른 기능군이라 '표현이 family를 구분하는지' 보기 좋습니다.

| family | 예시 |
|--------|------|
| Globin | 헤모글로빈 α/β/δ/γ, 미오글로빈 |
| Serine protease | 트립신, 키모트립시노겐, 엘라스타제, PSA … |
| Protein kinase | SRC, CDK1/2, ERK2, PKCδ |
| Cytochrome P450 | CYP3A4/2D6/2C9/1A2/2B6 |

In [ ]:
import io, time, urllib.request
import pandas as pd

# accession → family (모두 실제 UniProt 등록 단백질)
ACC2FAM = {
    "P69905":"Globin","P68871":"Globin","P02144":"Globin","P02042":"Globin","P69891":"Globin",
    "P07477":"SerineProtease","P17538":"SerineProtease","P08246":"SerineProtease","P07288":"SerineProtease","P08217":"SerineProtease",
    "P12931":"ProteinKinase","P24941":"ProteinKinase","P28482":"ProteinKinase","P06493":"ProteinKinase","Q05655":"ProteinKinase",
    "P08684":"CytochromeP450","P10635":"CytochromeP450","P11712":"CytochromeP450","P05177":"CytochromeP450","P20813":"CytochromeP450",
}

def fetch_fasta(acc):
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
    with urllib.request.urlopen(url, timeout=30) as r:
        txt = r.read().decode()
    lines = txt.splitlines()
    name = lines[0].split("|")[-1].split(" OS=")[0]
    seq = "".join(lines[1:])
    return name, seq

rows = []
for acc, fam in ACC2FAM.items():
    name, seq = fetch_fasta(acc)
    rows.append({"acc":acc, "family":fam, "name":name, "seq":seq, "length":len(seq)})
    time.sleep(0.2)  # NCBI/UniProt 예의상 간격

df = pd.DataFrame(rows)
print("단백질:", len(df), "| family별:", df["family"].value_counts().to_dict())
df[["acc","family","name","length"]]


## 2. 수작업 표현 (hand-crafted)

- **one-hot**: 잔기 하나를 20차원 벡터로. 서열 전체는 (길이×20) — 길이가 달라 **단백질 하나의 고정 벡터**로는 부적합.
- **아미노산 조성(AAC)**: one-hot을 서열 평균 → **20차원 고정 벡터**. 길이 무관, 간단·강건.
- **물성(ProtParam)**: 분자량·pI·GRAVY(평균 소수성)·불안정성·방향족성 등 **해석 가능한 소수 차원**.

In [ ]:
import numpy as np
from Bio.SeqUtils.ProtParam import ProteinAnalysis

AA = "ACDEFGHIKLMNPQRSTVWY"
AA_IDX = {a:i for i,a in enumerate(AA)}

def clean(seq):
    return "".join(c for c in seq.upper() if c in AA_IDX)  # 비표준(U,X,B,Z 등) 제거

def one_hot(seq):
    seq = clean(seq)
    M = np.zeros((len(seq), 20), dtype=float)
    for i, c in enumerate(seq):
        M[i, AA_IDX[c]] = 1.0
    return M  # (L, 20) — 시각화/설명용

def aac(seq):
    """아미노산 조성 = one-hot의 열평균 (20차원)."""
    M = one_hot(seq)
    return M.mean(axis=0) if len(M) else np.zeros(20)

def physchem(seq):
    """ProtParam 기반 해석가능 물성 (7차원)."""
    pa = ProteinAnalysis(clean(seq))
    helix, turn, sheet = pa.secondary_structure_fraction()
    return np.array([
        pa.molecular_weight(),
        pa.isoelectric_point(),
        pa.gravy(),                 # 평균 소수성
        pa.instability_index(),
        pa.aromaticity(),
        helix, sheet,
    ], dtype=float)

# one-hot 예시(첫 단백질)의 shape
demo = one_hot(df.iloc[0]["seq"])
print("one-hot 예시:", df.iloc[0]["name"], "→ shape", demo.shape, "(잔기 × 20)")

X_aac  = np.vstack([aac(s)      for s in df["seq"]])
X_phys = np.vstack([physchem(s) for s in df["seq"]])
print("AAC 행렬:", X_aac.shape, "| 물성 행렬:", X_phys.shape)


## 3. 학습된 표현 — ESM-2 임베딩

**ESM-2**(Lin et al., *Science* 2023)는 대규모 단백질 서열로 사전학습된 언어모델입니다.
서열을 넣으면 잔기별 문맥 벡터가 나오고, 이를 평균(mean-pool)하면 **단백질 한 개의 고정 임베딩**이 됩니다.
여기서는 가벼운 `esm2_t6_8M`(320차원)을 씁니다(CPU 가능). 더 큰 모델일수록 표현력↑.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

ESM_NAME = "facebook/esm2_t6_8M_UR50D"   # 더 크게: esm2_t12_35M_UR50D / esm2_t33_650M_UR50D
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(ESM_NAME)
esm = AutoModel.from_pretrained(ESM_NAME).to(device).eval()
print("ESM-2 로드:", ESM_NAME, "| device:", device)

@torch.no_grad()
def esm_embed(seq):
    enc = tok(clean(seq), return_tensors="pt", truncation=True, max_length=1022).to(device)
    out = esm(**enc).last_hidden_state[0]      # (L+2, H) : <cls> ... <eos>
    return out[1:-1].mean(0).cpu().numpy()     # 특수토큰 제외 mean-pool

X_esm = np.vstack([esm_embed(s) for s in df["seq"]])
print("ESM-2 임베딩 행렬:", X_esm.shape)


## 4. 비교 — 같은 과제, 다른 표현

세 표현(AAC 20d · 물성 7d · ESM-2 320d)을 **동일 조건**에서 비교합니다.

- **정량**: Leave-One-Out 1-NN family 분류 정확도 + 실루엣 점수(군집 분리도) — 모두 실측.
- **정성**: PCA 2D 투영에서 family가 얼마나 뭉치는지.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, silhouette_score

y = df["family"].values
reps = {"AAC (20d)": X_aac, "물성 (7d)": X_phys, "ESM-2 (320d)": X_esm}

summary = []
for name, X in reps.items():
    Xs = StandardScaler().fit_transform(X)
    pred = cross_val_predict(KNeighborsClassifier(n_neighbors=1), Xs, y, cv=LeaveOneOut())
    acc = accuracy_score(y, pred)
    sil = silhouette_score(Xs, y)      # family 라벨 기준 군집 분리도
    summary.append({"표현": name, "LOO 1-NN 정확도": round(acc,3), "실루엣": round(sil,3)})

res = pd.DataFrame(summary)
print("=== 표현별 family 구분 성능 (실측) ===")
res


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fams = sorted(df["family"].unique())
colors = {f:c for f,c in zip(fams, plt.cm.tab10.colors)}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, X) in zip(axes, reps.items()):
    Xs = StandardScaler().fit_transform(X)
    P = PCA(n_components=2, random_state=0).fit_transform(Xs)
    for f in fams:
        m = df["family"].values == f
        ax.scatter(P[m,0], P[m,1], c=[colors[f]], label=f, s=55, edgecolor="k", linewidth=0.4)
    ax.set_title(name); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
axes[-1].legend(fontsize=8, loc="best")
plt.suptitle("표현별 PCA 2D — family가 잘 뭉칠수록 좋은 표현", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ESM-2 임베딩의 코사인 유사도 히트맵 (family 순 정렬) — 대각 블록이 진하면 family 내 유사
from sklearn.metrics.pairwise import cosine_similarity

order = df.sort_values("family").index
Xo = X_esm[order]
labels = df.loc[order, "family"].values
S = cosine_similarity(StandardScaler().fit_transform(Xo))

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(S, cmap="viridis")
ax.set_title("ESM-2 임베딩 코사인 유사도 (family 순)")
# family 경계선
import numpy as np
bounds = np.where(labels[:-1] != labels[1:])[0] + 0.5
for b in bounds:
    ax.axhline(b, color="w", lw=1); ax.axvline(b, color="w", lw=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(df.loc[order,"acc"], rotation=90, fontsize=6)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(df.loc[order,"acc"], fontsize=6)
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()


## 5. 정리 — 언제 무엇을 쓰나

- **AAC / 물성(수작업)**: 빠르고 **해석 가능**, 데이터·계산 자원이 적을 때. 단, 서열 순서·문맥 정보는 버림.
- **ESM-2(학습된)**: 순서·진화 정보를 담아 표현력↑ — 보통 family 구분·전이학습에서 우수. 단, 모델 다운로드·계산 비용, 해석은 어려움.
- **구조 표현(contact/graph/surface, AlphaFold)**: 상호작용·결합부 등 **구조 의존 과제**에서 유리(본 실습 범위 밖).

> ⚠️ 위 정확도/실루엣은 **20개 소규모**에서의 실측치입니다. 값은 실행마다(모델·전처리) 달라질 수 있고, 정식 결론이 아니라 **표현법 비교 감각**을 위한 것입니다. 더 큰 ESM 모델(`esm2_t33_650M`)·더 많은 단백질로 확장해 보세요.

**참고문헌**
- Lin et al. *Evolutionary-scale prediction of atomic-level protein structure with a language model.* Science 379:1123 (2023) — ESM-2
- Elnaggar et al. *ProtTrans.* IEEE TPAMI (2021) — ProtT5
- Jumper et al. *Highly accurate protein structure prediction with AlphaFold.* Nature 596:583 (2021)
- Cock et al. *Biopython.* Bioinformatics 25:1422 (2009) — ProtParam